# Partie A simple — Quiver → table finale

Objectif : récupérer les transactions Congress Trading via Quiver, nettoyer le minimum nécessaire, et produire **une seule table finale** nommée `final_table`.

Ce notebook ne dépend d'aucun fichier de configuration externe et ne crée pas de dossiers `raw/bronze/silver`. La seule chose à faire est de coller le token Quiver dans la première cellule de code.

La table finale conserve les deux dates :
- `transaction_date` : date réelle du trade déclaré ;
- `disclosure_date` : date de dépôt/publication publique du PTR.

Pour la suite du projet, les backtests devront toujours entrer à partir de `disclosure_date`, pas `transaction_date`.


In [1]:
# ============================================================
# 1) CONFIGURATION — tout est ici, rien en variable d'environnement
# ============================================================

# Colle ton token Quiver ici.
# Important : évite de committer ce notebook sur GitHub avec le token rempli.
QUIVER_API_TOKEN = "e4f3cb20daf202dfc7bfae07af4ee8726a78affe"  # <- ex: "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

BASE_URL = "https://api.quiverquant.com"
ENDPOINT = "/beta/bulk/congresstrading"

# Endpoint Quiver Congress Trading
PARAMS = {
    "normalized": "true",
    "nonstock": "false",
    "version": "V2",
}

PAGE_SIZE = 1000

# Pour tester vite : mets MAX_PAGES = 2.
# Pour tout récupérer : laisse MAX_PAGES = None.
MAX_PAGES = None

SLEEP_SECONDS = 0.25
TIMEOUT_SECONDS = 60
MAX_RETRIES = 5

# Nettoyage final
DROP_MISSING_TICKER = True
DROP_MISSING_DATES = True
KEEP_ONLY_PURCHASE_SALE = True

# Convention pour les tranches ouvertes du type "> $50,000,000"
# Le document projet dit que cette convention doit être fixée explicitement.
OPEN_RANGE_MIDPOINT = 50_000_000.0


In [2]:
# ============================================================
# 2) IMPORTS
# ============================================================

import hashlib
import math
import re
import time
from typing import Any

import numpy as np
import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [3]:
# ============================================================
# 3) CLIENT API QUIVER
# ============================================================

def _extract_rows(payload: Any) -> list[dict]:
    """
    Quiver renvoie normalement une liste.
    Cette fonction accepte aussi des formes dict courantes, par prudence.
    """
    if payload is None:
        return []

    if isinstance(payload, list):
        return payload

    if isinstance(payload, dict):
        for key in ("results", "data", "items"):
            value = payload.get(key)
            if isinstance(value, list):
                return value

        # Certains endpoints pourraient retourner {"data": {"results": [...]}}
        data = payload.get("data")
        if isinstance(data, dict):
            for key in ("results", "items"):
                value = data.get(key)
                if isinstance(value, list):
                    return value

    raise TypeError(f"Format de réponse Quiver inattendu : {type(payload)}")


def quiver_get(endpoint: str, params: dict | None = None) -> list[dict]:
    """
    Appel GET Quiver avec authentification Bearer.
    """
    if not QUIVER_API_TOKEN or QUIVER_API_TOKEN.strip() == "":
        raise ValueError("Colle d'abord ton token Quiver dans QUIVER_API_TOKEN.")

    url = f"{BASE_URL}{endpoint}"
    headers = {"Authorization": f"Bearer {QUIVER_API_TOKEN.strip()}"}

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=TIMEOUT_SECONDS,
            )

            if response.status_code == 429:
                wait = min(60, 2 ** attempt)
                time.sleep(wait)
                continue

            if response.status_code in (401, 403):
                raise RuntimeError(
                    f"Erreur auth Quiver {response.status_code}. "
                    "Vérifie le token et le plan d'abonnement."
                )

            response.raise_for_status()
            return _extract_rows(response.json())

        except Exception as exc:
            last_error = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(min(60, 2 ** attempt))

    raise RuntimeError(f"Échec API Quiver après {MAX_RETRIES} essais : {last_error}")


def fetch_all_congress_trades() -> pd.DataFrame:
    """
    Télécharge toutes les pages Quiver Congress Trading.
    Ne sauvegarde rien sur disque : retourne directement un DataFrame brut.
    """
    rows_all: list[dict] = []
    page = 1

    while True:
        params = dict(PARAMS)
        params.update({"page": page, "page_size": PAGE_SIZE})

        rows = quiver_get(ENDPOINT, params=params)
        rows_all.extend(rows)

        if len(rows) < PAGE_SIZE:
            break

        if MAX_PAGES is not None and page >= MAX_PAGES:
            break

        page += 1
        time.sleep(SLEEP_SECONDS)

    return pd.DataFrame(rows_all)


In [4]:
# ============================================================
# 4) HELPERS DE NORMALISATION
# ============================================================

def first_existing(df: pd.DataFrame, candidates: list[str], default=None):
    """
    Retourne la première colonne existante parmi candidates.
    Si aucune n'existe, retourne une Series remplie avec default.
    """
    for col in candidates:
        if col in df.columns:
            return df[col]
    return pd.Series([default] * len(df), index=df.index)


def parse_date_series(s: pd.Series) -> pd.Series:
    """
    Parse une série de dates Quiver.
    Retourne une date normalisée au jour, sans timezone.
    """
    dt = pd.to_datetime(s, errors="coerce", utc=True)
    try:
        dt = dt.dt.tz_convert(None)
    except Exception:
        try:
            dt = dt.dt.tz_localize(None)
        except Exception:
            pass
    return dt.dt.normalize()


def clean_text_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NaT": pd.NA})
    )


def clean_ticker_series(s: pd.Series) -> pd.Series:
    out = clean_text_series(s).str.upper()
    out = out.str.replace(" ", "", regex=False)
    out = out.str.replace("/", ".", regex=False)
    out = out.replace({"-": pd.NA, "—": pd.NA, "N/A": pd.NA, "NA": pd.NA, "NONE": pd.NA})
    return out


def normalize_operation(x) -> str:
    if pd.isna(x):
        return "Other"

    t = str(x).strip().lower()

    if t in {"p", "purchase", "purchased", "buy", "bought"} or "purchase" in t:
        return "Purchase"

    if (
        t in {"s", "sale", "sold", "sell"}
        or "sale" in t
        or "sold" in t
        or "sell" in t
    ):
        return "Sale"

    if "exchange" in t:
        return "Exchange"

    return "Other"


def normalize_asset_type(x) -> str:
    if pd.isna(x):
        return "Unknown"

    t = str(x).strip().lower()

    if t in {"st", "cs"}:
        return "Stock"
    if "common stock" in t or t == "stock":
        return "Stock"
    if "option" in t:
        return "Option"
    if "etf" in t or "fund" in t or t in {"ot"}:
        return "Fund/ETF"
    if "bond" in t or "note" in t:
        return "Bond"

    return "Other"


def money_to_float(x) -> float | None:
    if x is None or pd.isna(x):
        return None

    s = str(x)
    s = s.replace("$", "").replace(",", "").replace("USD", "")
    s = s.replace("usd", "").replace("+", "")
    s = s.strip()

    if s == "":
        return None

    # garde seulement le premier nombre trouvé
    m = re.search(r"[-+]?\d*\.?\d+", s)
    if not m:
        return None

    try:
        return float(m.group(0))
    except Exception:
        return None


def parse_amount_range(x) -> pd.Series:
    """
    Parse les montants Quiver/PTR.
    Cas possibles :
    - "$1,001 - $15,000"
    - "$1,001 to $15,000"
    - "> $50,000,000"
    - "$50,000,000+"
    - "1001" / "$1001" si Quiver ne donne qu'une borne basse
    """
    if x is None or pd.isna(x):
        return pd.Series({
            "amount_lower_bound": np.nan,
            "amount_upper_bound": np.nan,
            "amount_midpoint": np.nan,
            "amount_method": "missing",
        })

    raw = str(x).strip()
    s = raw.lower().replace("\u2013", "-").replace("\u2014", "-").replace("–", "-").replace("—", "-")

    is_open = any(token in s for token in [">", "over", "more than"]) or s.strip().endswith("+")

    nums = re.findall(r"\$?\s*[\d,]+(?:\.\d+)?", raw)
    values = [money_to_float(n) for n in nums]
    values = [v for v in values if v is not None and not math.isnan(v)]

    if len(values) >= 2:
        lo, hi = min(values[0], values[1]), max(values[0], values[1])
        return pd.Series({
            "amount_lower_bound": lo,
            "amount_upper_bound": hi,
            "amount_midpoint": (lo + hi) / 2,
            "amount_method": "range_midpoint",
        })

    if len(values) == 1:
        lo = values[0]
        if is_open:
            return pd.Series({
                "amount_lower_bound": lo,
                "amount_upper_bound": np.nan,
                "amount_midpoint": OPEN_RANGE_MIDPOINT,
                "amount_method": "open_range_convention",
            })

        return pd.Series({
            "amount_lower_bound": lo,
            "amount_upper_bound": np.nan,
            "amount_midpoint": lo,
            "amount_method": "lower_bound_only",
        })

    return pd.Series({
        "amount_lower_bound": np.nan,
        "amount_upper_bound": np.nan,
        "amount_midpoint": np.nan,
        "amount_method": "unparsed",
    })


def make_trade_id(row: pd.Series) -> str:
    cols = [
        "declarant_name",
        "bioguide_id",
        "chamber",
        "ticker",
        "company_name",
        "operation_type",
        "transaction_date",
        "disclosure_date",
        "amount_raw",
        "description",
    ]
    payload = "|".join("" if pd.isna(row.get(c, pd.NA)) else str(row.get(c)) for c in cols)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:24]


In [5]:
# ============================================================
# 5) NORMALISATION QUIVER → TABLE FINALE
# ============================================================

FINAL_COLUMNS = [
    "trade_id",
    "source",
    "declarant_name",
    "bioguide_id",
    "chamber",
    "party",
    "district",
    "state",
    "ticker",
    "company_name",
    "asset_type",
    "asset_type_group",
    "operation_type",
    "operation_group",
    "transaction_date",
    "disclosure_date",
    "disclosure_lag_days",
    "amount_raw",
    "amount_lower_bound",
    "amount_upper_bound",
    "amount_midpoint",
    "amount_method",
    "status",
    "subholding",
    "description",
    "comments",
    "quiver_upload_time",
    "last_modified",
]


def normalize_quiver_congress_trades(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Convertit la réponse Quiver V2/V1 en table finale unique.
    """
    if raw_df.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)

    df = raw_df.copy()

    out = pd.DataFrame(index=df.index)

    # Identité déclarant
    out["declarant_name"] = clean_text_series(first_existing(df, ["Name", "Representative", "Senator"]))
    out["bioguide_id"] = clean_text_series(first_existing(df, ["BioGuideID"]))
    out["chamber"] = clean_text_series(first_existing(df, ["Chamber", "House"]))
    out["party"] = clean_text_series(first_existing(df, ["Party", "party"]))
    out["district"] = clean_text_series(first_existing(df, ["District"]))
    out["state"] = clean_text_series(first_existing(df, ["State"]))

    # Instrument
    out["ticker"] = clean_ticker_series(first_existing(df, ["Ticker"]))
    out["company_name"] = clean_text_series(first_existing(df, ["Company"]))
    out["asset_type"] = clean_text_series(first_existing(df, ["TickerType", "Type"]))
    out["asset_type_group"] = out["asset_type"].apply(normalize_asset_type)

    # Opération
    out["operation_type"] = clean_text_series(first_existing(df, ["Transaction"]))
    out["operation_group"] = out["operation_type"].apply(normalize_operation)

    # Dates : V2 = Traded / Filed ; V1 = TransactionDate ou Date / ReportDate
    out["transaction_date"] = parse_date_series(first_existing(df, ["Traded", "TransactionDate", "Date"]))
    out["disclosure_date"] = parse_date_series(first_existing(df, ["Filed", "ReportDate"]))

    out["quiver_upload_time"] = pd.to_datetime(
        first_existing(df, ["Quiver_Upload_Time"]),
        errors="coerce",
        utc=True,
    ).dt.tz_convert(None)

    out["last_modified"] = pd.to_datetime(
        first_existing(df, ["last_modified"]),
        errors="coerce",
        utc=True,
    ).dt.tz_convert(None)

    # Montants
    out["amount_raw"] = clean_text_series(first_existing(df, ["Trade_Size_USD", "Range", "Amount"]))
    amount_cols = out["amount_raw"].apply(parse_amount_range)
    out = pd.concat([out, amount_cols], axis=1)

    # Champs texte complémentaires
    out["status"] = clean_text_series(first_existing(df, ["Status"]))
    out["subholding"] = clean_text_series(first_existing(df, ["Subholding"]))
    out["description"] = clean_text_series(first_existing(df, ["Description"]))
    out["comments"] = clean_text_series(first_existing(df, ["Comments"]))

    out["source"] = "quiver_bulk_congresstrading"

    # Délai de divulgation
    out["disclosure_lag_days"] = (
        out["disclosure_date"] - out["transaction_date"]
    ).dt.days

    # trade_id stable interne
    out["trade_id"] = out.apply(make_trade_id, axis=1)

    # Déduplication stricte
    out = out.drop_duplicates("trade_id").copy()

    # Filtres minimaux pour une table exploitable
    if DROP_MISSING_TICKER:
        out = out[out["ticker"].notna()].copy()

    if DROP_MISSING_DATES:
        out = out[
            out["transaction_date"].notna()
            & out["disclosure_date"].notna()
        ].copy()

    if KEEP_ONLY_PURCHASE_SALE:
        out = out[out["operation_group"].isin(["Purchase", "Sale"])].copy()

    # Colonnes finales et tri
    for col in FINAL_COLUMNS:
        if col not in out.columns:
            out[col] = pd.NA

    out = out[FINAL_COLUMNS].sort_values(
        ["disclosure_date", "transaction_date", "declarant_name", "ticker"],
        na_position="last",
    ).reset_index(drop=True)

    return out


In [6]:
# ============================================================
# 6) EXÉCUTION — récupération Quiver puis table finale
# ============================================================

raw_quiver_table = fetch_all_congress_trades()
final_table = normalize_quiver_congress_trades(raw_quiver_table)

# Une seule sortie : la table finale.
display(final_table)


,trade_id,source,declarant_name,bioguide_id,chamber,party,district,state,ticker,company_name,asset_type,asset_type_group,operation_type,operation_group,transaction_date,disclosure_date,disclosure_lag_days,amount_raw,amount_lower_bound,amount_upper_bound,amount_midpoint,amount_method,status,subholding,description,comments,quiver_upload_time,last_modified
0,bbeebccefd740d5b175c876d,quiver_bulk_congresstrading,Mr. Bob Gibbs,G000563,House,Republican,7.0,Ohio,TKR,TIMKEN COMPANY,<NA>,Unknown,Purchase,Purchase,2013-12-18,2014-01-03,16,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
1,5f88cb242285140eda378bed,quiver_bulk_congresstrading,Mr. Bob Gibbs,G000563,House,Republican,7.0,Ohio,TEG,"INTEGRYS ENERGY GROUP, INC.",<NA>,Unknown,Sale,Sale,2014-01-03,2014-01-03,0,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
2,215ba5ef206f98cd12033cc9,quiver_bulk_congresstrading,Mr. Bill Flores,F000461,House,Republican,17.0,Texas,NGLS,TARGA RESOURCES PARTNERS LP COMMON UNITS REPRE...,<NA>,Unknown,Purchase,Purchase,2013-12-02,2014-01-07,36,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
3,01ae688b03323c29c69de794,quiver_bulk_congresstrading,Mr. Bill Flores,F000461,House,Republican,17.0,Texas,BBEP,"BREITBURN ENERGY PARTNERS, L.P. - COMMON UNITS...",<NA>,Unknown,Sale,Sale,2013-12-04,2014-01-07,34,"$250,001 - $500,000",250001.0,500000.0,375000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
4,aa633d45b49ce067c24cb5fa,quiver_bulk_congresstrading,Mr. Bill Flores,F000461,House,Republican,17.0,Texas,WMB,"TRANSACTION TYPE.) WILLIAMS COMPANIES, INC.",<NA>,Unknown,Sale,Sale,2013-12-04,2014-01-07,34,"$250,001 - $500,000",250001.0,500000.0,375000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98965,6bd7ac7485d3c00d730a62b3,quiver_bulk_congresstrading,Jared Moskowitz,M001217,House,Democratic,23.0,Florida,GILD,"GILEAD SCIENCES, INC. - COMMON STOCK",ST,Stock,Purchase,Purchase,2026-05-06,2026-06-19,44,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,MORGAN STANLEY ACTIVE ASSETS (5),<NA>,<NA>,2026-06-22,2026-06-22
98966,e0db04693cf9c42f2c698582,quiver_bulk_congresstrading,Nancy Pelosi,P000197,House,Democratic,11.0,California,INTC,INTEL CORPORATION - COMMON STOCK,OP,Other,Purchase,Purchase,2026-05-29,2026-06-23,25,"$1,000,001 - $5,000,000",1000001.0,5000000.0,3000000.5,range_midpoint,NEW,<NA>,PURCHASED 200 CALL OPTIONS WITH A STRIKE PRICE...,<NA>,2026-06-24,2026-06-24
98967,47b12d233639224f58644879,quiver_bulk_congresstrading,Nancy Pelosi,P000197,House,Democratic,11.0,California,UBER,"UBER TECHNOLOGIES, INC. COMMON STOCK",OP,Other,Purchase,Purchase,2026-05-29,2026-06-23,25,"$500,001 - $1,000,000",500001.0,1000000.0,750000.5,range_midpoint,NEW,<NA>,PURCHASED 200 CALL OPTIONS WITH A STRIKE PRICE...,<NA>,2026-06-24,2026-06-24
98968,e85d24afaeef758aca303508,quiver_bulk_congresstrading,Dwight Evans,E000296,House,Democratic,3.0,Pennsylvania,GD,GENERAL DYNAMICS CORPORATION COMMON STOCK,ST,Stock,Sale,Sale,2026-06-10,2026-06-24,14,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,CETERA,<NA>,<NA>,2026-06-25,2026-06-25


## Colonnes de `final_table`

La table finale contient uniquement les champs utiles pour la suite :

```text
trade_id
source
declarant_name
bioguide_id
chamber
party
district
state
ticker
company_name
asset_type
asset_type_group
operation_type
operation_group
transaction_date
disclosure_date
disclosure_lag_days
amount_raw
amount_lower_bound
amount_upper_bound
amount_midpoint
amount_method
status
subholding
description
comments
quiver_upload_time
last_modified
```

Pour la phase B, on utilisera surtout `ticker`, `operation_group`, `transaction_date`, `disclosure_date`, `disclosure_lag_days`, `amount_midpoint`, `declarant_name`, `bioguide_id`, `chamber` et `party`.
